# 05 - Generate the synthetic data

**Owner:** Daria  |  **Output:** every table of the data model in `data/synthetic/`, plus `app/data/expert_demo.json` for the expert screens

The risks, countries, triggers and outcomes are **real** ACAPS data. Everything about experts is **synthetic**: registrations, approvals per country and domain, anonymity, documents, display names, swipes, to-do items, agree/disagree answers, comments, AI-drafted sources, likes, forecasts and suggestions.

The generating code is in `src/synthetic_data.py`. This notebook runs it, checks the result and saves it. Change the settings in section 1 and run everything again; the same seed always gives the same data.

## 0. Setup

In [1]:
import sys, json, importlib
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

def find_repo_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "src" / "synthetic_data.py").exists():
            return folder
    raise FileNotFoundError("Could not find src/synthetic_data.py - put it in the src/ folder of the repo")

ROOT = find_repo_root()
sys.path.append(str(ROOT / "src"))
import risk_tools, synthetic_data
importlib.reload(risk_tools); importlib.reload(synthetic_data)   # picks up edits without restarting
from synthetic_data import CONFIG, CATEGORIES, build_all, validate, save_all, expert_demo_bundle
print("Repo root:", ROOT)

Repo root: C:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth


## 1. Settings
Change these if needed. `match_weights` are the starting weights for Valeri's algorithm (country and domain 1.0, country only 0.6, domain only 0.4).

In [2]:
cfg = dict(CONFIG)
cfg.update({
    "today": "2026-11-06",
    "seed": 42,
    "n_experts": 80,
    "prob_scheme": "infographic",   # or "pdf" - confirm with ACAPS
    "demo_risk": "R1043",
})
EXPORT   = ROOT / "data" / "risk-radar-export.csv"
OUT_DIR  = ROOT / "data" / "synthetic"
APP_FILE = ROOT / "app" / "data" / "expert_demo.json"
CLEAN_OLD = True   # remove the files the old notebooks 03 and 04 made
cfg

{'today': '2026-11-06',
 'seed': 42,
 'n_experts': 80,
 'prob_scheme': 'infographic',
 'match_weights': {'country_and_domain': 1.0,
  'country_only': 0.6,
  'domain_only': 0.4},
 'anonymity_mix': {'known_to_acaps': 0.3,
  'verified_anonymous': 0.4,
  'verified_deleted': 0.3},
 'status_mix': {'approved': 0.88, 'pending': 0.07, 'rejected': 0.05},
 'default_horizon_months': 2,
 'demo_risk': 'R1043'}

## 2. Build all tables

In [3]:
tables, truth = build_all(EXPORT, cfg)
pd.DataFrame({"rows": {k: len(v) for k, v in tables.items()}})

,rows
categories,15
risks,139
risk_countries,152
risk_categories,457
triggers,269
experts,86
expert_identity,86
expert_countries,105
expert_domains,156
application_documents,177


## 3. Checks
### 3a. Do all links between tables work, and are the rules kept?
Every ID that points to another table must exist there; nobody changes their opinion more than once; no two experts share a name on the same risk; experts who chose "verified, then deleted" have no personal data left. An empty list means everything is fine.

In [4]:
problems = validate(tables)
problems or "No problems found"

'No problems found'

### 3b. Categories (domains)
The export has no category column, so categories come from keywords in each risk's title and triggers. Look through this list: if a risk is in the wrong category, add or change a keyword in `CATEGORY_KEYWORDS` in `src/synthetic_data.py`.

In [5]:
r = tables["risks"]
print(r["category"].value_counts().to_string(), "\n")
print("Card colours of active risks:", r["urgency_colour"].value_counts().to_dict(), "\n")
r[["risk_id", "title", "category", "category_image"]].head(25)

category
Political crisis            32
Cross-border insecurity     25
Infrastructure failure      19
Displacement                15
Flood                       12
Economic crisis / prices     9
Armed conflict               6
Cyclone / storm              4
Food insecurity              4
Earthquake / landslide       4
Disease outbreak             4
Drought                      3
Aid and funding cuts         1
Fallback (no match)          1 

Card colours of active risks: {'Red': 12, 'Yellow': 11, 'Orange': 5} 



,risk_id,title,category,category_image
0,R0353,Taliban attacks on provincial capitals lead to civil war violence ...,Armed conflict,cat-conflict
1,R0990,"Unsafe return of Afghan nationals from Pakistan causes food, shelt...",Aid and funding cuts,cat-funding
2,R1015,La Niña in Afghanistan worsens drought and leads to widespread agr...,Flood,cat-flood
3,R1062,Continued attacks between Pakistan and Afghanistan increase protec...,Cross-border insecurity,cat-border
4,R1067,Escalating clashes between Interim Taliban Authority forces and ar...,Cross-border insecurity,cat-border
5,R1056,Resource-related clashes and political tensions in Tindouf camps r...,Political crisis,cat-political
6,R0371,Heavy rains in late August 2022 cause floods that affect previousl...,Cyclone / storm,cat-storm
7,R1020,Intensifying political violence approaching the February 2026 elec...,Infrastructure failure,cat-infrastructure
8,R1039,Prolonged fuel shortages in Bangladesh worsen food security and di...,Infrastructure failure,cat-infrastructure
9,R0510,High public debt has reduced state capacity to provide public serv...,Political crisis,cat-political


In [6]:
# risks that fell back to category 0: check whether a keyword is missing
r.loc[r["category_id"] == 0, ["risk_id", "title"]]

,risk_id,title
57,R0089,Deterioration of humanitarian situation in Mali


### 3c. Coverage: does every active risk have enough experts and answers?

In [7]:
m = tables["expert_risk_matches"]
active = r[r["status"] == "Active"]
cov = active[["risk_id", "country", "category", "urgency"]].copy()
cov["eligible_experts"] = cov["risk_id"].map(m.groupby("risk_id").size()).fillna(0).astype(int)
cov["local_and_topic_experts"] = cov["risk_id"].map(m[m["match_type"] == "country_and_domain"].groupby("risk_id").size()).fillna(0).astype(int)
cov["answers"] = cov["risk_id"].map(tables["stances"].groupby("risk_id")["expert_id"].nunique()).fillna(0).astype(int)
cov["comments"] = cov["risk_id"].map(tables["comments"].groupby("risk_id").size()).fillna(0).astype(int)
print("Active risks with fewer than 3 answers (group view would stay hidden):", int((cov["answers"] < 3).sum()))
cov.sort_values("answers")

Active risks with fewer than 3 answers (group view would stay hidden): 0


,risk_id,country,category,urgency,eligible_experts,local_and_topic_experts,answers,comments
30,R1045,Ethiopia,Flood,urgent,17,2,8,1
75,R1038,Niger,Displacement,very_urgent,19,2,12,2
61,R1042,Mali,Infrastructure failure,very_urgent,31,2,16,9
19,R1059,Colombia,Political crisis,very_urgent,30,2,16,7
120,R1068,Syria,Political crisis,watch,24,3,16,4
35,R1071,Indonesia,Flood,very_urgent,27,2,17,7
81,R1061,Palestine,Political crisis,urgent,32,3,18,6
55,R1046,Madagascar,Flood,watch,36,3,19,6
76,R1074,Niger,Political crisis,watch,33,4,19,6
89,R1058,Philippines,Flood,very_urgent,34,3,24,9


### 3d. Experts, approval and anonymity

In [8]:
ex = tables["experts"]
print(ex["status"].value_counts().to_string(), "\n")
print(ex["anonymity_level"].value_counts().to_string(), "\n")
print("Country requests:", tables["expert_countries"]["status"].value_counts().to_dict())
print("Domain requests: ", tables["expert_domains"]["status"].value_counts().to_dict())
print("Anonymity changes in settings:", int((tables["anonymity_history"]["changed_in"] == "settings").sum()))

status
approved    78
pending      5
rejected     3 

anonymity_level
verified_deleted      35
verified_anonymous    26
known_to_acaps        25 

Country requests: {'approved': 93, 'pending': 6, 'denied': 6}
Domain requests:  {'approved': 132, 'denied': 15, 'pending': 9}
Anonymity changes in settings: 6


In [9]:
# privacy check: who still has personal data stored?
ident = tables["expert_identity"].merge(ex[["expert_id", "status", "anonymity_level"]])
ident.groupby(["anonymity_level", "status"]).agg(
    experts=("expert_id", "size"),
    with_name=("full_name", lambda s: int(s.notna().sum())),
    with_contact_email=("contact_email", lambda s: int(s.notna().sum())),
    with_recovery_code=("recovery_code_hash", lambda s: int(s.notna().sum())))

experts  with_name  with_contact_email  with_recovery_code
anonymity_level    status                                                              
known_to_acaps     approved       24         24                  24                  24
                   rejected        1          1                   1                   1
verified_anonymous approved       22         22                  22                  22
                   pending         2          2                   2                   2
                   rejected        2          2                   2                   2
verified_deleted   approved       32          0                   0                  32
                   pending         3          3                   3                   3

### 3e. What experts did

In [10]:
print("Card swipes:", tables["card_swipes"]["result"].value_counts().to_dict())
print("To-do items:", tables["todo_items"]["status"].value_counts().to_dict())
st = tables["stances"]
print("Answers:", st[st["version"] == 1]["stance"].value_counts().to_dict(), "| changed opinion:", int(st["changed_opinion"].sum()))
print("Comments:", tables["comments"]["kind"].value_counts().to_dict(), "| with AI-drafted source:", int(tables["comments"]["ai_draft_used"].sum()))
fc = tables["forecasts"].merge(tables["forecast_questions"][["question_id", "is_backtest"]])
print("Forecasts on open questions:", int((~fc["is_backtest"]).sum()), "| on past risks (backtest):", int(fc["is_backtest"].sum()))
print("Suggestions:", tables["suggestions"].groupby(["type", "status"]).size().to_dict())
print("Triggers:", tables["triggers"]["source"].value_counts().to_dict())

Card swipes: {'todo': 410, 'done': 365, 'skipped': 100}
To-do items: {'done': 250, 'open': 160}
Answers: {'agree': 436, 'disagree': 290} | changed opinion: 55
Comments: {'answer': 243, 'changed_opinion': 55} | with AI-drafted source: 61
Forecasts on open questions: 509 | on past risks (backtest): 1657
Suggestions: {('missing_trigger', 'accepted'): 3, ('missing_trigger', 'new'): 3, ('new_risk', 'accepted'): 4, ('new_risk', 'new'): 1, ('new_risk', 'rejected'): 5}
Triggers: {'acaps': 266, 'expert_suggestion': 3}


### 3f. The demo expert's deck

In [11]:
bundle = expert_demo_bundle(tables, truth, cfg)
print("Demo expert:", bundle["expert"]["expert_id"], "|", bundle["expert"]["anonymity_level"])
print("Countries:", bundle["expert"]["countries"])
print("Domains:", bundle["expert"]["domains"])
pd.DataFrame(bundle["deck"])[["risk_id", "title", "country", "category", "label", "days_left", "match_type"]]

Demo expert: E007 | verified_deleted
Countries: [{'country': 'Chad', 'status': 'approved'}, {'country': np.str_('Nigeria'), 'status': 'approved'}]
Domains: [{'category': 'Cross-border insecurity', 'status': 'approved'}, {'category': 'Political crisis', 'status': 'approved'}, {'category': 'Earthquake / landslide', 'status': 'approved'}]


,risk_id,title,country,category,label,days_left,match_type
0,R1037,Escalating violence between the Sudan People's Liberation Movement...,South Sudan,Cross-border insecurity,Very urgent,4,domain_only
1,R1059,Peace talk end and military offensive against armed groups in Colo...,Colombia,Political crisis,Very urgent,8,domain_only
2,R1043,Escalating cross-border insecurity linked to the Sudan conflict in...,Chad,Cross-border insecurity,Very urgent,9,country_and_domain
3,R1069,Military operations to secure the Dawei Special Economic Zone caus...,Myanmar,Cross-border insecurity,Very urgent,11,domain_only
4,R1062,Continued attacks between Pakistan and Afghanistan increase protec...,Afghanistan,Cross-border insecurity,Very urgent,19,domain_only
5,R1066,Escalating insecurity across the Lake Chad Basin increases displac...,"Chad, Niger, Cameroon, Nigeria",Cross-border insecurity,Very urgent,28,country_and_domain
6,R1067,Escalating clashes between Interim Taliban Authority forces and ar...,Afghanistan,Cross-border insecurity,Urgent,3,domain_only
7,R1072,Sustained military pressure in Cabo Delgado drives Islamic State M...,Mozambique,Cross-border insecurity,Urgent,4,domain_only
8,R1061,Permanent military exclusion zone east of the Yellow Line before t...,Palestine,Political crisis,Urgent,5,domain_only
9,R1073,Accelerated implementation of the E1 settlement plan results in de...,Palestine,Infrastructure failure,Urgent,17,domain_only


## 4. Save

In [12]:
if CLEAN_OLD:
    for name in ["responses.csv", "panel.csv", "scoreboard.csv", "demo_R1043_panel.json"]:
        p = OUT_DIR / name
        if p.exists():
            p.unlink()
            print("removed old file", name)

saved = save_all(tables, truth, OUT_DIR)
APP_FILE.parent.mkdir(parents=True, exist_ok=True)
APP_FILE.write_text(json.dumps(bundle, ensure_ascii=False, indent=2), encoding="utf-8")
print(len(saved), "tables saved in", OUT_DIR)
print("App file:", APP_FILE)

removed old file responses.csv
removed old file panel.csv
removed old file scoreboard.csv
removed old file demo_R1043_panel.json
25 tables saved in C:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth\data\synthetic
App file: C:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth\app\data\expert_demo.json


## Notes for the team
- **Valeri** builds the algorithms on these tables. The ones he needs most: `expert_risk_matches` (match type and starting weight), `stances` (answers, changed opinion), `forecasts` with `forecast_questions` (open questions and past ones with real outcomes), `comments`, `comment_likes` and `suggestions` (contribution).
- **`_simulation/expert_truth.csv`** holds the hidden skill, bias, noise and activity of each synthetic expert. Use it only to **test** whether an algorithm finds the good experts; the app must never read it.
- **The demo expert** has no activity on active risks on purpose, so the deck starts full. Their past forecasts exist, so a rank can be shown.
- **Designers** read `app/data/expert_demo.json`: the deck (already sorted by urgency), details per risk with triggers and the forecast question, each risk's community, and the read-only discussions.
- **Account recovery** works like a crypto wallet: every expert gets a 12-word secret recovery code at registration and is shown it once. Only a salted hash of the code is stored, so ACAPS cannot read it; without the code a forgotten password cannot be reset. The plain codes of the synthetic experts are only in `_simulation/expert_truth.csv`, so the team can test the recovery screen.
- **Triggers:** most come from ACAPS (the export, `source = acaps`). Expert suggestions only join the list once ACAPS accepts them (`source = expert_suggestion`); the app never shows which expert suggested one.
- **Colours:** every risk is yellow or above. There is no green.
- Emails are made up (`expertNNN@example.org`) and names are "Synthetic Expert NNN". No real person is in this data.